# ESG Keyword Extraction — Evaluation

Evaluate extracted concepts against manually labelled ESG examples. The starter benchmark is intentionally small; expand it with reviewed corporate disclosures before drawing performance conclusions.

In [ ]:
import pandas as pd
from utils import keywords_tfidf, keywords_rake, keywords_textrank

gold = pd.read_csv('data/labeled_sentences.csv')
gold

In [ ]:
def evaluate(df, method, k=10):
    rows=[]
    for _, r in df.iterrows():
        expected = {x.strip().lower() for x in str(r['keyword']).split('|') if x.strip() and x != 'nan'}
        if not expected: continue
        predicted = {x[0].lower() for x in method(r['sentence'], k)}
        hits = len(expected & predicted)
        p = hits / max(len(predicted), 1)
        rec = hits / len(expected)
        f1 = 2*p*rec/max(p+rec, 1e-12)
        rows.append((p, rec, f1))
    if not rows: return pd.Series({'precision@k':0,'recall':0,'f1':0})
    a = pd.DataFrame(rows, columns=['precision@k','recall','f1'])
    return a.mean()

comparison = pd.DataFrame({
    'TF-IDF': evaluate(gold, keywords_tfidf),
    'RAKE': evaluate(gold, keywords_rake),
    'TextRank': evaluate(gold, keywords_textrank),
}).T
comparison

## Evaluation discipline

Do not report the starter benchmark as a production-quality score. Build a larger gold-standard set from real ESG disclosures, split it into development and held-out evaluation data, and use the held-out set to compare model variants. Include error analysis for generic corporate terms, abbreviations, multi-word concepts, negation and ambiguous governance/environmental language.